In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shayanshahid997/yellow-taxi-trip-record-of-january-2024")

print("Path to dataset files:", path)

C:\Users\kamek\PycharmProjects\portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\kamek\.cache\kagglehub\datasets\shayanshahid997\yellow-taxi-trip-record-of-january-2024\versions\1


In [2]:
import polars as pl
import os

In [3]:
parquet_path = os.path.join(path, "yellow_tripdata_2024-01.parquet")

In [4]:
lf = pl.scan_parquet(parquet_path)


In [5]:
df = lf.collect()

In [6]:
print(df.shape)
df.head(5)

(2964624, 19)


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


# Operation showcase

### Time-aware dynamic windows

In [7]:
(
    lf
    .sort('tpep_pickup_datetime')
    .group_by_dynamic(
        "tpep_pickup_datetime",
        every="1h",
        period="1h",
        closed="left"
    )
    .agg(
        pl.col("fare_amount").mean().alias("avg_fare")
    )
).collect()


tpep_pickup_datetime,avg_fare
datetime[ns],f64
2002-12-31 22:00:00,0.0
2009-01-01 00:00:00,50.6
2009-01-01 23:00:00,24.7
2023-12-31 23:00:00,14.41
2024-01-01 00:00:00,18.977327
…,…
2024-01-31 20:00:00,16.2719
2024-01-31 21:00:00,16.671163
2024-01-31 22:00:00,17.579284


### AVG trip distance per vendor

In [8]:
(
    lf
    .select(['VendorID', 'trip_distance'])
    .group_by('VendorID')
    .agg(pl.col('trip_distance').mean().alias('avg_trip_distance'))
    .sort('avg_trip_distance')
    .collect()
)

VendorID,avg_trip_distance
i32,f64
1,2.974737
2,3.872493
6,11.346846


### ABG tip distance per vendor - SQL
In polars we can use SQL syntax to perform operations as well.

In [9]:
(
    lf
    .sql(
    """
    select VendorID, avg(trip_distance) as avg_trip_distance
    from self
    group by VendorID
    order by avg_trip_distance
    """
    )
    .collect()
)

VendorID,avg_trip_distance
i32,f64
1,2.974737
2,3.872493
6,11.346846


# Multi table operations
## Show correlation between demand and temperature
Show difference of taxi demand with correlation to temperature below month average, and precipitation above month average.

#### Load DFs

Taxi data

In [10]:
# Download all 2024 taxi data data

path = kagglehub.dataset_download("sygnation/nyc-yellow-taxi-records-2024")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\kamek\.cache\kagglehub\datasets\sygnation\nyc-yellow-taxi-records-2024\versions\1


In [11]:
taxi_ldf = pl.scan_parquet(os.path.join(path, "*.parquet"), try_parse_hive_dates=True)
schema = dict(taxi_ldf.collect_schema().items())
schema['tpep_pickup_datetime'] = pl.Datetime("us")
schema['tpep_dropoff_datetime'] = pl.Datetime("us")

taxi_ldf = pl.scan_parquet(os.path.join(path, "*.parquet"), schema=schema, cast_options=pl.ScanCastOptions(datetime_cast='nanosecond-downcast'))

In [12]:
taxi_ldf.collect().head(5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


Weather data

In [13]:
# Loading latest weather data for 2024 in New York City
weather_lf = pl.scan_csv("data/nyc-2024-weather.csv")
weather_lf.collect().head(5)

STATION,NAME,DATE,AWND,PGTM,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDF2,WDF5,WSF2,WSF5,WT01,WT02,WT03,WT06,WT08
str,str,str,f64,str,f64,f64,f64,str,i64,i64,str,str,f64,f64,str,str,str,str,str
"""USW00094728""","""NY CITY CENTRAL PARK, NY US""","""2024-01-01""",3.36,null,0.03,0.0,0.0,null,47,35,""" 270""",""" 220""",8.9,16.1,null,null,null,null,null
"""USW00094728""","""NY CITY CENTRAL PARK, NY US""","""2024-01-02""",4.03,null,0.0,0.0,0.0,null,42,29,""" 300""",""" 320""",10.1,16.1,null,null,null,null,null
"""USW00094728""","""NY CITY CENTRAL PARK, NY US""","""2024-01-03""",4.92,null,0.0,0.0,0.0,null,43,34,""" 300""",""" 320""",10.1,15.0,null,null,null,null,null
"""USW00094728""","""NY CITY CENTRAL PARK, NY US""","""2024-01-04""",7.61,null,0.0,0.0,0.0,null,45,28,""" 310""",""" 300""",19.9,30.0,null,null,null,null,null
"""USW00094728""","""NY CITY CENTRAL PARK, NY US""","""2024-01-05""",7.16,null,0.0,0.0,0.0,null,37,26,""" 300""",""" 290""",16.1,23.9,null,null,null,null,null


#### Prepare dataframes

In [14]:
taxi_ldf_per_day = (
    taxi_ldf
    .filter(
        pl.col('tpep_pickup_datetime').dt.year() == 2024
    )
    .sort('tpep_pickup_datetime')
    .group_by_dynamic(
        "tpep_pickup_datetime",
        every="1d",
    )
    .agg([
        pl.len().alias("total_trips"),
        pl.sum('trip_distance').alias("daily_distance"),
        pl.mean('trip_distance').alias("avg_daily_distance"),
        pl.sum('fare_amount').alias("daily_fare_amount"),
        pl.mean('fare_amount').alias("avg_daily_fare_amount"),
    ])
    .rename({
        "tpep_pickup_datetime": "date"
    })
)

In [15]:
weather_lf_ny = (
    weather_lf
    .with_columns(
        pl.col('DATE').str.strptime(pl.Datetime, "%Y-%m-%d").cast(pl.Datetime("us"))
    )
    .select([
        pl.col('DATE').alias("date"),
        ((((pl.col("TMAX") + pl.col("TMIN")) / 2) - 32) * 5/9).round(2).alias("avg_temp"),  # Calc mean and convert to C
        (pl.col('PRCP') * 25.4).round(2).alias('precipitation')  # Convert inch to mm
    ])
)

#### Calculations
Transform DF to present avg daily distance per weekday

In [16]:
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def cast_weekday_to_weekname(x):
    return weekday_names[x - 1]

combined_df = (
    taxi_ldf_per_day
    .join(
        weather_lf_ny,
        on='date',
        how='left'
    )
    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
    )
    .sort('weekday')
    .with_columns(
        pl.col('weekday').map_elements(cast_weekday_to_weekname, return_dtype=pl.String)
    )
    .collect()
)

In [17]:

avg_week_distance = (
    combined_df
    .group_by('weekday', maintain_order=True)
    .agg([
        pl.col('avg_daily_distance').mean(),
    ])
    .with_columns(
        pl.lit("Average").alias('source')
    )
)

avg_week_distance_lower_temps = (
    combined_df
    # Filter temps than lower avg
    .join(
        combined_df.select(pl.col('avg_temp').mean().alias('year_avg_temp')), how='cross'
    )
    .filter(
        pl.col('avg_temp') < pl.col('year_avg_temp')
    )
    .group_by('weekday', maintain_order=True)
    .agg([
        pl.col('avg_daily_distance').mean(),
    ])
    .with_columns(
        pl.lit("Lower temps").alias('source')
    )
)

avg_week_distance_higher_precipitation = (
    combined_df
    .join(
        combined_df.select(pl.col('precipitation').mean().alias('year_precipitation')), how='cross'
    )
    .filter(
        pl.col('precipitation') < pl.col('year_precipitation')
    )
    .group_by('weekday', maintain_order=True)
    .agg([
        pl.col('avg_daily_distance').mean(),
    ])
    .with_columns(
        pl.lit("Higher precipitation").alias('source')
    )
)


### Results

In [18]:
import plotly.express as px

chart = (
    px.line(
        pl.concat([
            avg_week_distance,
            avg_week_distance_lower_temps,
            avg_week_distance_higher_precipitation
        ]),
        x="weekday",
        y="avg_daily_distance",
        color="source",
        markers=True,
        title="Average Daily Distance per Weekday by Weather Condition"
    )
)

chart.update_layout(
    title={
        'text': "Average Daily Distance per Weekday by Weather Condition",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=22)
    },
    xaxis_title="Weekday",
    yaxis_title="Average Distance (miles)",
    legend_title_text="Condition",
    font=dict(
        family="Arial",
        size=14
    ),
)

chart.show()



### Analysis

The graph shows Average Daily Distance per Weekday by Weather Condition:

- Average weather and higher precipitation conditions result in similar distances throughout the week, peaking on Sunday at over 5.6 miles.
- Lower temperature days consistently show reduced activity, especially from Tuesday to Saturday, with the lowest point on Saturday (~4.45 miles).
- Across all conditions, Sunday tends to have the highest distance, suggesting increased activity on weekends regardless of weather.
- Lower temperatures appear to have a greater negative impact on daily distance than higher precipitation.